# Gemini Vision Example

**HFA AI Training — Module 6: Vision Language Models**

Google Gemini is currently the best Vision Language Model (VLM), especially
for tasks involving video, large images, and spatial reasoning. It was built
multimodal from the ground up — vision isn't an add-on, it's core to the model.

This notebook demonstrates:
1. Loading an image from a local file
2. Loading an image from a URL
3. Asking Gemini to analyze and describe an image
4. Asking follow-up questions about the same image
5. Extracting structured data (JSON) from an image

**Get a free API key at:** https://aistudio.google.com/apikey

## Install Dependencies

In [ ]:
!pip install google-generativeai httpx pillow

## Imports and Configuration

Load the required libraries and configure the Gemini API key from environment variables.
Never hard-code API keys — always use environment variables.

In [ ]:
import os
import json
import httpx
from pathlib import Path

import google.generativeai as genai
from PIL import Image

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Always load API keys from environment variables — never hard-code them.
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise EnvironmentError(
        "GOOGLE_API_KEY not set. Run: export GOOGLE_API_KEY='your-key'\n"
        "Get a free key at https://aistudio.google.com/apikey"
    )

# Configure the SDK with our API key.
genai.configure(api_key=GOOGLE_API_KEY)

# Use the latest Gemini model with vision capabilities.
# Gemini 2.0 Flash is fast, capable, and cost-effective for vision tasks.
MODEL_NAME = "gemini-2.0-flash"

## Example 1: Analyze an Image from a URL

Download an image from a URL and ask Gemini to analyze it.
This is the simplest way to get started — just point Gemini at an image
and ask a question about it.

In [ ]:
def analyze_image_from_url(image_url: str, question: str) -> str:
    """
    Download an image from a URL and ask Gemini to analyze it.

    This is the simplest way to get started — just point Gemini at an image
    and ask a question about it.

    Args:
        image_url: URL of the image to analyze.
        question: What you want to know about the image.

    Returns:
        Gemini's text response about the image.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 1: Analyze an image from a URL")
    print(f"{'='*60}")
    print(f"Image URL: {image_url}")
    print(f"Question:  {question}")

    # Download the image bytes from the URL.
    response = httpx.get(image_url)
    image_bytes = response.content

    # Create the Gemini model instance.
    model = genai.GenerativeModel(MODEL_NAME)

    # Send the image and question to Gemini.
    # Gemini accepts images as raw bytes, PIL Images, or file uploads.
    gemini_response = model.generate_content(
        [
            {
                "mime_type": "image/jpeg",
                "data": image_bytes,
            },
            question,
        ]
    )

    print(f"\nGemini's Analysis:\n{gemini_response.text}")
    return gemini_response.text

## Example 2: Analyze a Local Image File

Load an image from a local file and ask Gemini about it.
Uses PIL (Pillow) to open the image, which Gemini's SDK accepts directly.

In [ ]:
def analyze_local_image(image_path: str, question: str) -> str:
    """
    Load an image from a local file and ask Gemini about it.

    Uses PIL (Pillow) to open the image, which Gemini's SDK accepts directly.

    Args:
        image_path: Path to a local image file (jpg, png, etc.).
        question: What you want to know about the image.

    Returns:
        Gemini's text response.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 2: Analyze a local image file")
    print(f"{'='*60}")
    print(f"Image path: {image_path}")
    print(f"Question:   {question}")

    # Open the image with PIL. Gemini's SDK can accept PIL Image objects directly.
    image = Image.open(image_path)

    model = genai.GenerativeModel(MODEL_NAME)

    # Pass the PIL Image directly — the SDK handles the conversion.
    gemini_response = model.generate_content([image, question])

    print(f"\nGemini's Analysis:\n{gemini_response.text}")
    return gemini_response.text

## Example 3: Multi-Turn Conversation About an Image

Have a multi-turn conversation with Gemini about a single image.
This demonstrates follow-up questions — like a real conversation where
you keep asking about the same photo.

In [ ]:
def conversation_about_image(image_url: str) -> None:
    """
    Have a multi-turn conversation with Gemini about a single image.

    This demonstrates follow-up questions — like a real conversation where
    you keep asking about the same photo.

    Args:
        image_url: URL of the image to discuss.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 3: Multi-turn conversation about an image")
    print(f"{'='*60}")

    # Download the image.
    response = httpx.get(image_url)
    image_bytes = response.content

    model = genai.GenerativeModel(MODEL_NAME)

    # Start a chat session. This maintains conversation history so Gemini
    # remembers the image and previous questions.
    chat = model.start_chat()

    # First question — send the image along with it.
    questions = [
        "What do you see in this image? Describe it in detail.",
        "What style of architecture or design is shown?",
        "If this were a real estate listing, what would you highlight as key selling points?",
    ]

    # First message includes the image.
    print(f"\nYou: {questions[0]}")
    response = chat.send_message(
        [
            {"mime_type": "image/jpeg", "data": image_bytes},
            questions[0],
        ]
    )
    print(f"Gemini: {response.text}\n")

    # Follow-up questions — no need to re-send the image, Gemini remembers it.
    for question in questions[1:]:
        print(f"You: {question}")
        response = chat.send_message(question)
        print(f"Gemini: {response.text}\n")

## Example 4: Structured Data Extraction from an Image

Ask Gemini to extract structured JSON data from an image.

This is incredibly powerful for real-world applications:
- Extract line items from a receipt photo
- Pull data from a business card
- Read a floor plan and output room dimensions

This is "OCR on steroids" — not just reading text, but UNDERSTANDING
the document and organizing the data.

In [ ]:
def extract_structured_data(image_url: str) -> dict:
    """
    Ask Gemini to extract structured JSON data from an image.

    This is incredibly powerful for real-world applications:
    - Extract line items from a receipt photo
    - Pull data from a business card
    - Read a floor plan and output room dimensions

    This is "OCR on steroids" — not just reading text, but UNDERSTANDING
    the document and organizing the data.

    Args:
        image_url: URL of the image (e.g., a receipt, form, document).

    Returns:
        Parsed dictionary of extracted data.
    """
    print(f"\n{'='*60}")
    print("EXAMPLE 4: Structured data extraction from an image")
    print(f"{'='*60}")

    response = httpx.get(image_url)
    image_bytes = response.content

    model = genai.GenerativeModel(MODEL_NAME)

    # The key is in the prompt: ask for specific JSON structure.
    # Be explicit about what fields you want extracted.
    prompt = """Analyze this image and extract all relevant information into
    a structured JSON format.

    If this is a property/real estate photo, use this structure:
    {
        "property_type": "house/condo/land/commercial",
        "architectural_style": "description of style",
        "condition": "excellent/good/fair/poor",
        "visible_features": ["feature1", "feature2"],
        "estimated_bedrooms": null or number,
        "estimated_bathrooms": null or number,
        "exterior_materials": ["material1", "material2"],
        "landscaping": "description",
        "notable_details": ["detail1", "detail2"],
        "potential_concerns": ["concern1", "concern2"]
    }

    If this is a document, receipt, or form, extract the relevant fields.

    Return ONLY the JSON — no additional text or markdown formatting."""

    gemini_response = model.generate_content(
        [
            {"mime_type": "image/jpeg", "data": image_bytes},
            prompt,
        ]
    )

    # Parse the JSON response.
    raw_text = gemini_response.text.strip()

    # Gemini sometimes wraps JSON in markdown code blocks — strip those.
    if raw_text.startswith("```"):
        raw_text = raw_text.split("\n", 1)[1]  # Remove first line (```json)
        raw_text = raw_text.rsplit("```", 1)[0]  # Remove last ``` block

    try:
        structured_data = json.loads(raw_text)
        print(f"\nExtracted structured data:")
        print(json.dumps(structured_data, indent=2))
        return structured_data
    except json.JSONDecodeError:
        print(f"\nGemini returned text (not strict JSON):\n{gemini_response.text}")
        return {"raw_response": gemini_response.text}

## Run All Examples

Execute all the Gemini vision examples using a sample public domain house photo.

In [ ]:
print("=" * 60)
print("  GEMINI VISION EXAMPLES")
print("  Google Gemini — The Best VLM for Vision Tasks")
print("=" * 60)

# A sample image URL — a public domain house photo.
# Replace with any image URL you'd like to analyze.
sample_image_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "1/19/Weatherboard_house_in_Coorparoo%2C_Queensland.jpg/"
    "1280px-Weatherboard_house_in_Coorparoo%2C_Queensland.jpg"
)

### Example 1: Simple image analysis from URL

In [ ]:
analyze_image_from_url(
    image_url=sample_image_url,
    question="Describe this property in detail. What are its key features?",
)

### Example 2: Local image analysis

Uncomment and provide a real file path to test.

In [ ]:
# analyze_local_image(
#     image_path="/path/to/your/image.jpg",
#     question="What do you see in this image?",
# )

### Example 3: Multi-turn conversation

In [ ]:
conversation_about_image(sample_image_url)

### Example 4: Structured data extraction

In [ ]:
extract_structured_data(sample_image_url)

In [ ]:
print("\n" + "=" * 60)
print("  All examples complete!")
print("=" * 60)